In [22]:
#=== HomeGuard Security System ===
# Client: The Peterson Family
# Author: Gordan
# Description: The HomeGuard system processes risks through 3 segments:
# Security Alerts, Safety Alerts and Comfort Notifications & Patterns
#======================================================================


import random
from datetime import datetime


# ---------------------------------------------------------
# Sensor class: represents one sensor (motion, door, temp, smoke)
# Combines data (location, type, value) with behavior (read, check)
# ---------------------------------------------------------
class Sensor:
    def __init__(self, sensor_id, location, sensor_type, normal_range=None):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.normal_range = normal_range  # only used by temperature sensors
        self.current_value = None

    def read(self):
        """Generates a new random reading appropriate to this sensor's type."""
        if self.type == "temperature":
            self.current_value = random.randint(30, 90)
        elif self.type == "door":
            self.current_value = random.choice(["CLOSED", "OPENED"])
        elif self.type == "smoke":
            self.current_value = random.choices(["CLEAR", "DETECTED"], weights=[95, 5])[0]
        elif self.type == "motion":
            self.current_value = random.choices(["No activity", "DETECTED"], weights=[90, 10])[0]
        return self.current_value

    def isAbnormal(self):
        """Returns True if the current reading is outside normal parameters."""
        if self.type == "temperature":
            low, high = self.normal_range
            return self.current_value < low or self.current_value > high
        elif self.type == "door":
            return self.current_value == "OPENED"
        elif self.type == "smoke":
            return self.current_value == "DETECTED"
        elif self.type == "motion":
            return self.current_value == "DETECTED"
        return False

    def reset(self):
        """Resets the sensor's value to None (used when re-arming the system)."""
        self.current_value = None


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------
def log_event(message):
    """Records an event with a timestamp, simulating a notification log."""
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")


def trigger_alert(sensor, system_mode):
    """
    Decides, based on sensor type and current system mode, whether to
    print an alert. Uses if/else logic to route each sensor type to
    the right severity/category.
    """
    if sensor.type == "door" and system_mode == "AWAY":
        print("[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!")
        log_event("Sending notification to homeowner...")

    elif sensor.type == "smoke":
        print(f"[ALERT!] 🚨 HIGH: SAFETY: Smoke detected in {sensor.location}!")
        log_event("Sending notification to homeowner...")

    elif sensor.type == "temperature":
        print(f"[ALERT!] ⚠️ MEDIUM: SAFETY: {sensor.location} temperature out of range: "
              f"{sensor.current_value}°F")

    elif sensor.type == "motion" and system_mode == "AWAY":
        print(f"[ALERT!] 🚨 MEDIUM: SECURITY: Motion detected in {sensor.location}!")


# ---------------------------------------------------------
# Main simulation loop
# ---------------------------------------------------------
def run_simulation(system_mode="AWAY", iterations=3):
    """Creates the sensor list and runs the simulation for a set number
    of time steps, printing readings and any triggered alerts."""

    sensor_list = [
        Sensor("S1", "Living Room", "motion"),
        Sensor("S2", "Front Door", "door"),
        Sensor("S3", "Kitchen", "temperature", (60, 75)),
        Sensor("S4", "Bedroom", "smoke"),
    ]

    print("=== HomeGuard Security System ===")
    print(f"Time: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Mode: {system_mode}\n")

    for i in range(iterations):
        if i > 0:
            print(f"\nTime: {datetime.now().strftime('%H:%M:%S')}")

        for sensor in sensor_list:
            sensor.read()
            print(f"[READING] {sensor.location} {sensor.type.capitalize()}: {sensor.current_value}")

            if sensor.isAbnormal():
                trigger_alert(sensor, system_mode)


# ---------------------------------------------------------
# Entry point
# ---------------------------------------------------------
if __name__ == "__main__":
    run_simulation(system_mode="HOME", iterations=3)

=== HomeGuard Security System ===
Time: 09:44:48
Mode: HOME

[READING] Living Room Motion: No activity
[READING] Front Door Door: OPENED
[READING] Kitchen Temperature: 44
[ALERT!] ⚠️ MEDIUM: SAFETY: Kitchen temperature out of range: 44°F
[READING] Bedroom Smoke: CLEAR

Time: 09:44:48
[READING] Living Room Motion: DETECTED
[READING] Front Door Door: OPENED
[READING] Kitchen Temperature: 61
[READING] Bedroom Smoke: CLEAR

Time: 09:44:48
[READING] Living Room Motion: No activity
[READING] Front Door Door: CLOSED
[READING] Kitchen Temperature: 53
[ALERT!] ⚠️ MEDIUM: SAFETY: Kitchen temperature out of range: 53°F
[READING] Bedroom Smoke: CLEAR
